# Cross-Session Continuity — GRPO Training

**Design:** Single-step GRPO. LLM writes handoff notes only.
S1 and S2 are scripted. Reward = 40% note quality + 60% S2 test pass rate.

| Mode | Model | Time |
|------|-------|------|
| `--fast` | Qwen2.5-0.5B | ~10 min |
| `--full` | Qwen2.5-Coder-7B | ~60 min |

In [ ]:
# ── Install ───────────────────────────────────────────────────────────────
%%capture
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl>=0.12 transformers datasets accelerate bitsandbytes scipy matplotlib openenv-core
print('done')

In [ ]:
# ── Clone repo ────────────────────────────────────────────────────────────
import os, sys
if 'google.colab' in sys.modules:
    !git clone https://huggingface.co/spaces/Aswini-Kumar/cross-session-continuity-env /content/env
    os.chdir('/content/env')
    sys.path.insert(0, '/content/env')
print('CWD:', os.getcwd())

In [ ]:
# ── Run training ──────────────────────────────────────────────────────────
# Change --fast to --full for the 7B submission run
!python training/grpo_train.py --fast

In [ ]:
# ── Display plots ─────────────────────────────────────────────────────────
from IPython.display import Image, display
import os
for f in ['loss_curve.png','reward_curve.png','baseline_vs_trained.png',
          'ablation_comparison.png','difficulty_breakdown.png','handoff_diff_over_epochs.png']:
    p = f'plots/{f}'
    if os.path.exists(p):
        print(f'--- {f} ---')
        display(Image(p))

In [ ]:
# ── Push model to Hub (optional) ─────────────────────────────────────────
import os
HF_TOKEN = os.environ.get('HF_TOKEN', '')
if HF_TOKEN:
    from unsloth import FastLanguageModel
    # reload the trained model
    model, tok = FastLanguageModel.from_pretrained('results/grpo_checkpoints')
    model.push_to_hub_merged(
        'Aswini-Kumar/cross-session-continuity-model',
        tok, save_method='merged_16bit', token=HF_TOKEN,
    )
    print('Pushed to Hub')
else:
    print('Add HF_TOKEN to Colab Secrets to push model')